In [ ]:

import pandas as pd
import numpy as np
import trimesh
import numpy as np
import pytz
from read_mesh import facenormal_angle,powerratio_iam,shadowing_factor
from pv_index import pv_perf_index
from sun import get_sun_direction
from datetime import datetime

Read out the car

In [3]:
mesh=trimesh.load_mesh('models/car.stl')


read out thesampled points

In [ ]:
# --- about point.csv ---
# a file with points sampled in 15-minutes intervall.
# the car starts in Darwin at 10:00 local time, and travels
# perfectly south, at 85kph, for 8 hours.
df=pd.read_csv('/Users/nils/ace/workspace/Solar/data/points.csv')

# convert to UTC
df['time'] = pd.to_datetime(
    df['time'],
    format='%Y-%m-%d %H:%M:%S'
)
df_time_acst = df['time'].dt.tz_localize('Australia/Darwin', ambiguous='infer')
df['time_utc'] = df_time_acst.dt.tz_convert('UTC')

#insight
df.head()

,Lat,Long,altitude,time,time_utc
0,-12.45,130.83,100,2025-09-15 10:00:00,2025-09-15 00:30:00+00:00
1,-12.64,130.83,100,2025-09-15 10:15:00,2025-09-15 00:45:00+00:00
2,-12.83,130.83,100,2025-09-15 10:30:00,2025-09-15 01:00:00+00:00
3,-13.02,130.83,100,2025-09-15 10:45:00,2025-09-15 01:15:00+00:00
4,-13.21,130.83,100,2025-09-15 11:00:00,2025-09-15 01:30:00+00:00


**Table/Matrix with corresponing performance values** 
- rows: Facettes
- columns: point in time

This table contains the weight of each facette of the mesh for every point in time. It could forexamplke be used to identy rows/facettes with the best correlation, (areas that perform and behave equally) this might be interesting to create a layout lateron.


In [ ]:

res = pd.DataFrame()
for index, row in df.iterrows():

    sun_dir = get_sun_direction(
        row['Lat'],
        row['Long'],
        row['altitude'],
        row['time_utc']
        )
    angles = facenormal_angle(mesh,sun_dir)
    shadow_mask = shadowing_factor(mesh, -1*sun_dir)
    power_weight = powerratio_iam(angles)
    power_weight[shadow_mask]=0
    res[f'{index}']=power_weight

res.to_csv('out.csv') #consider a faster dataformat for further use.